# Environment and pinned assets

Formal DiffJev workflow; see AGENTS.md and prompt.md.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/DiffJev")
if not (ROOT / "AGENTS.md").exists():
    subprocess.run(
        ["git", "clone", "https://github.com/BoldHu/DiffJev.git", str(ROOT)],
        check=True,
    )

os.chdir(ROOT)
os.environ["DIFFJEV_ROOT"] = str(ROOT)

In [ ]:
import os, sys, gc
from pathlib import Path
ROOT = Path(os.environ.get('DIFFJEV_ROOT', Path.cwd())).resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'AGENTS.md').exists(), 'Set DIFFJEV_ROOT to the checked-out repository'
sys.path.insert(0, str(ROOT / 'src'))
from diffjev import workflows as w
from diffjev.config import load_yaml_config
cfg = load_yaml_config(str(ROOT / 'configs/workflow.yaml'))
SEED, DEVICE = cfg['seed'], cfg['device']


Use Python 3.12 for the formal environment. Installation and downloads below are explicit opt-in operations. Upload complete pinned snapshots to the paths in configs/models.yaml and configs/data.yaml; include download_provenance.json. No model weights are fetched by importing DiffJev.

In [ ]:
INSTALL = False
if INSTALL:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT/'requirements-lock.txt')])

In [ ]:
from diffjev.assets import environment_record, model_specs
w.save_json(ROOT/'outputs/formal/environment.json', environment_record())
print(environment_record())
for name,spec in model_specs().items():
    print(name, spec['revision'], ROOT/spec['local_dir'])

In [ ]:
# Optional large downloads; edit only on the intended experiment machine.
DOWNLOAD = False
if DOWNLOAD:
    import subprocess
    subprocess.check_call([sys.executable, str(ROOT/'scripts/download_assets.py'), '--all-core', '--yes'])

Before Exp 5, expose exactly one A100 80GB with CUDA_VISIBLE_DEVICES. Larger training runs may be scheduled independently on the three GPUs. This code does not silently shard model layers across devices.